# Notebook 1 (v2): ML-декодеры для Surface Code — Улучшенные модели

**Что изменилось по сравнению с v1:**
- `ResidualMLP`: residual-блоки + hand-crafted syndrome features (вес, чётность, per-round веса)
- `SurfaceCodeCNN`: правильная (d-1)×d укладка ancilla-кубитов вместо sqrt-паддинга
- `AncillaTransformer`: 2D пространственные + временны́е positional embeddings, соответствующие реальной геометрии кода
- `FocalLoss`: борьба с дисбалансом классов (α=0.75, γ=2)
- Больше данных: 200k train вместо 50k
- Правильный LR: 1e-4 (не 3e-4), больше эпох (60), pos_weight как дополнительная защита


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

from qec_ml.utils.config import QECConfig, NoiseConfig, TrainingConfig
from qec_ml.data import SyndromeGenerator, SyndromeDatasetTorch, SyndromeSpatialDatasetTorch, make_dataloaders
from qec_ml.decoders import MWPMDecoder, MLDecoderWrapper
from qec_ml.models import (
    MLPDecoder, CNNDecoder,              # v1 legacy
    ResidualMLP, SurfaceCodeCNN,         # v2 improved
    FocalLoss,                           # loss for imbalanced classes
    SyndromeTransformer,                 # v1 transformer
    SpatialTemporalTransformer,          # v1 ST-transformer
    AncillaTransformer,                  # v2: 2D spatial pos encoding
)
from qec_ml.utils.training import Trainer
from qec_ml.benchmarks import (
    BenchmarkRunner,
    plot_decoder_comparison, plot_ler_vs_noise,
    plot_ler_vs_distance, plot_training_curves, plot_confusion,
)
from torch.utils.data import DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
plt.rcParams['figure.dpi'] = 120
print(f'Device: {DEVICE}')


## 1. Генерация данных (200k шотов)

In [ ]:
DISTANCE = 5
ROUNDS = 5
NOISE_P = 0.01

cfg = QECConfig(
    distance=DISTANCE,
    noise=NoiseConfig(model='depolarizing', p=NOISE_P, rounds=ROUNDS),
    n_samples_train=200_000,
    n_samples_val=20_000,
    n_samples_test=20_000,
    seed=SEED,
)

gen = SyndromeGenerator(cfg)
full_data = gen.generate(n_samples=240_000)
train_raw, val_raw, test_raw = full_data.split(train=0.833, val=0.083)

print(f'Syndrome length: {cfg.syndrome_length}')
print(f'Train: {len(train_raw)} | Val: {len(val_raw)} | Test: {len(test_raw)}')

# Class balance
pos_frac = train_raw.logical_errors.mean()
neg_frac = 1 - pos_frac
pos_weight_val = neg_frac / pos_frac
print(f'Class balance — no error: {neg_frac:.3f}, logical error: {pos_frac:.3f}')
print(f'pos_weight for BCE: {pos_weight_val:.3f}')

# Compute AncillaTransformer geometry
H = DISTANCE - 1    # ancilla rows
W = DISTANCE        # ancilla cols
N_ANC_GRID = H * W  # = d*(d-1)
L = cfg.syndrome_length
print(f'Ancilla grid: {H} x {W} = {N_ANC_GRID} (stim detectors total: {L // ROUNDS})')


## 2. Подготовка датасетов и DataLoader'ов

In [ ]:
# Flat datasets (MLP, Transformers)
train_ds = SyndromeDatasetTorch(train_raw, augment=True)
val_ds   = SyndromeDatasetTorch(val_raw)
test_ds  = SyndromeDatasetTorch(test_raw)

# 2D spatial datasets (CNN)
train_ds_2d = SyndromeSpatialDatasetTorch(train_raw, augment=True)
val_ds_2d   = SyndromeSpatialDatasetTorch(val_raw)
test_ds_2d  = SyndromeSpatialDatasetTorch(test_raw)
SIDE = train_ds_2d.side
PADDED_LEN = train_ds_2d.padded_length
print(f'2D grid side: {SIDE}, padded length: {PADDED_LEN}')

# Shared training config template
def make_cfg(model_type, epochs=60, lr=1e-4, bs=512):
    return TrainingConfig(
        model_type=model_type,
        epochs=epochs,
        batch_size=bs,
        learning_rate=lr,
        warmup_epochs=5,
        scheduler='cosine',
        early_stopping_patience=12,
        gradient_clip=1.0,
        device=DEVICE,
    )

def make_loaders(ds_train, ds_val, ds_test, bs=512):
    kw = dict(batch_size=bs, num_workers=0, pin_memory=False)
    return (
        DataLoader(ds_train, shuffle=True, **kw),
        DataLoader(ds_val,   shuffle=False, **kw),
        DataLoader(ds_test,  shuffle=False, **kw),
    )

train_loader, val_loader, test_loader = make_loaders(train_ds, val_ds, test_ds)
train_loader_2d, val_loader_2d, test_loader_2d = make_loaders(train_ds_2d, val_ds_2d, test_ds_2d)


## 3. MWPM Baseline

In [ ]:
import time
circuit = gen.get_circuit()
mwpm = MWPMDecoder(cfg).build(circuit)
t0 = time.perf_counter()
mwpm_preds = mwpm.decode_batch(test_raw.syndromes)
mwpm_time_us = (time.perf_counter() - t0) / len(test_raw) * 1e6
mwpm_ler = float(np.mean(mwpm_preds != test_raw.logical_errors))
print(f'MWPM LER: {mwpm_ler:.4f}  ({mwpm_time_us:.1f} µs/shot)')


## 4. Улучшенные ML-модели

### 4.0 Вспомогательные функции

`FocalLoss` с `alpha=0.75` штрафует предсказание мажоритарного класса.  
`pos_weight` в BCE — дополнительная страховка от коллапса к мажоритарному классу.

In [ ]:
# Функция потерь: Focal Loss (борьба с дисбалансом классов)
focal_loss = FocalLoss(alpha=0.75, gamma=2.0)

# Альтернатива: weighted BCE
pw = torch.tensor([pos_weight_val])
weighted_bce = nn.BCEWithLogitsLoss(pos_weight=pw)

print(f'FocalLoss(alpha=0.75, gamma=2.0)')
print(f'Weighted BCE pos_weight={pos_weight_val:.2f}')

# Быстрый тест forward pass для каждой модели
def check_model(model, input_shape, name):
    model.eval()
    x = torch.zeros(*input_shape)
    with torch.no_grad():
        out = model(x)
    n = sum(p.numel() for p in model.parameters())
    print(f'{name:30s} | output: {tuple(out.shape)} | params: {n:,}')
    return n


### 4.1 ResidualMLP — MLP с residual-блоками и syndrome features

In [ ]:
# ── ResidualMLP ──────────────────────────────────────────────────────
rmlp = ResidualMLP(
    syndrome_length=L,
    d_model=256,
    n_blocks=8,
    dropout=0.1,
    rounds=ROUNDS,
)
n_rmlp = check_model(rmlp, (4, L), 'ResidualMLP')

rmlp_trainer = Trainer(rmlp, make_cfg('mlp', epochs=60, lr=1e-4),
                       loss_fn=focal_loss)
rmlp_history = rmlp_trainer.fit(train_loader, val_loader)
rmlp_trainer.load_best()
rmlp_result = rmlp_trainer.evaluate(test_loader)
print(f'ResidualMLP test LER: {rmlp_result["logical_error_rate"]:.4f}')


### 4.2 SurfaceCodeCNN — CNN с физически правильной укладкой ancilla

In [ ]:
# ── SurfaceCodeCNN ───────────────────────────────────────────────────
# Входной размер: (B, ROUNDS, d-1, d) = (B, 5, 4, 5)
# Это соответствует реальной геометрии Z-стабилизаторов
sc_cnn = SurfaceCodeCNN(
    distance=DISTANCE,
    rounds=ROUNDS,
    base_channels=64,
    n_blocks=5,
    dropout=0.1,
)
n_cnn = check_model(sc_cnn, (4, L), 'SurfaceCodeCNN')
print(f'  Grid: (B, {ROUNDS}, {DISTANCE-1}, {DISTANCE}) — ancilla rows x cols')

# CNN принимает flat syndrome и делает reshape внутри
cnn_trainer = Trainer(sc_cnn, make_cfg('cnn', epochs=60, lr=1e-4),
                      loss_fn=focal_loss)
cnn_history = cnn_trainer.fit(train_loader, val_loader)  # flat loader!
cnn_trainer.load_best()
cnn_result = cnn_trainer.evaluate(test_loader)
print(f'SurfaceCodeCNN test LER: {cnn_result["logical_error_rate"]:.4f}')


### 4.3 AncillaTransformer — Transformer с 2D+temporal positional encoding

**Ключевое отличие от v1**: каждому ancilla-токену присваивается тройка (row, col, round),
соответствующая реальным координатам на решётке поверхностного кода.
Это позволяет attention-механизму сразу учитывать пространственную близость стабилизаторов.

In [ ]:
# ── AncillaTransformer ───────────────────────────────────────────────
at = AncillaTransformer(
    distance=DISTANCE,
    rounds=ROUNDS,
    d_model=128,
    n_heads=8,
    n_layers=6,
    dropout=0.1,
)
n_at = check_model(at, (4, L), 'AncillaTransformer')
print(f'  Tokens: {ROUNDS} rounds × {H}×{W} ancilla = {ROUNDS*H*W} per sample')

at_trainer = Trainer(at, make_cfg('transformer', epochs=60, lr=1e-4),
                     loss_fn=focal_loss)
at_history = at_trainer.fit(train_loader, val_loader)
at_trainer.load_best()
at_result = at_trainer.evaluate(test_loader)
print(f'AncillaTransformer test LER: {at_result["logical_error_rate"]:.4f}')


### 4.4 SyndromeTransformer (v1, переобученный с FocalLoss)

In [ ]:
# Тот же v1, но теперь с Focal Loss + правильным LR
tr_v1 = SyndromeTransformer(
    syndrome_length=L,
    d_model=128,
    n_heads=8,
    n_layers=6,
    dropout=0.1,
    use_cls_token=True,
)
n_trv1 = check_model(tr_v1, (4, L), 'SyndromeTransformer-v1')

trv1_trainer = Trainer(tr_v1, make_cfg('transformer', epochs=60, lr=1e-4),
                       loss_fn=focal_loss)
trv1_history = trv1_trainer.fit(train_loader, val_loader)
trv1_trainer.load_best()
trv1_result = trv1_trainer.evaluate(test_loader)
print(f'SyndromeTransformer-v1 test LER: {trv1_result["logical_error_rate"]:.4f}')


### 4.5 Ensemble — усреднение вероятностей моделей

Простой soft-voting ensemble часто на 5–15% лучше любой отдельной модели.

In [ ]:
class EnsembleDecoder:
    """Soft-voting ensemble of multiple MLDecoderWrappers."""
    def __init__(self, wrappers, name='Ensemble'):
        self._wrappers = wrappers
        self._name = name

    @property
    def name(self): return self._name

    @torch.no_grad()
    def decode_batch(self, syndromes):
        probas = []
        for w in self._wrappers:
            p = w.predict_proba(syndromes)
            probas.append(p)
        avg = np.mean(probas, axis=0)
        return (avg > 0.5).astype(np.uint8)

# Wrapper factories
rmlp_w = MLDecoderWrapper(rmlp,  'ResidualMLP',         device=DEVICE)
cnn_w  = MLDecoderWrapper(sc_cnn,'SurfaceCodeCNN',      device=DEVICE)
at_w   = MLDecoderWrapper(at,    'AncillaTransformer',  device=DEVICE)
trv1_w = MLDecoderWrapper(tr_v1, 'SyndromeTransformer', device=DEVICE)

ensemble = EnsembleDecoder([rmlp_w, cnn_w, at_w, trv1_w])
ens_preds = ensemble.decode_batch(test_raw.syndromes)
ens_ler = np.mean(ens_preds != test_raw.logical_errors)
print(f'Ensemble LER: {ens_ler:.4f}  (MWPM: {mwpm_ler:.4f})')


## 5. Сравнительный анализ всех моделей

In [ ]:
import time

def eval_decoder(name, decode_fn, syndromes, labels):
    t0 = time.perf_counter()
    preds = decode_fn(syndromes)
    ms = (time.perf_counter() - t0) / len(labels) * 1000
    ler = float(np.mean(preds != labels))
    return {'decoder': name, 'LER': ler, 'accuracy': 1-ler, 'ms/shot': ms}

rows = []
rows.append(eval_decoder('MWPM', mwpm.decode_batch, test_raw.syndromes, test_raw.logical_errors))
rows.append(eval_decoder('ResidualMLP',        rmlp_w.decode_batch,  test_raw.syndromes, test_raw.logical_errors))
rows.append(eval_decoder('SurfaceCodeCNN',     cnn_w.decode_batch,   test_raw.syndromes, test_raw.logical_errors))
rows.append(eval_decoder('SyndromeTransf-v1', trv1_w.decode_batch,  test_raw.syndromes, test_raw.logical_errors))
rows.append(eval_decoder('AncillaTransf',      at_w.decode_batch,   test_raw.syndromes, test_raw.logical_errors))
rows.append(eval_decoder('Ensemble',           ensemble.decode_batch, test_raw.syndromes, test_raw.logical_errors))

df = pd.DataFrame(rows).sort_values('LER')
df['vs_MWPM'] = (df['LER'] / mwpm_ler).round(3)
df['n_params'] = df['decoder'].map({
    'MWPM': 0, 'ResidualMLP': n_rmlp, 'SurfaceCodeCNN': n_cnn,
    'SyndromeTransf-v1': n_trv1, 'AncillaTransf': n_at, 'Ensemble': 0
}).fillna(0).astype(int)
print(df.to_string(index=False))
print('\nvs_MWPM < 1.0  →  модель лучше MWPM')


### 5.1 Кривые обучения

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
histories = [
    (rmlp_history,  'ResidualMLP'),
    (cnn_history,   'SurfaceCodeCNN'),
    (trv1_history,  'SyndromeTransformer-v1'),
    (at_history,    'AncillaTransformer'),
]
for i, (hist, name) in enumerate(histories):
    ep = range(1, len(hist.train_loss)+1)
    axes[0,i].plot(ep, hist.train_loss, label='Train')
    axes[0,i].plot(ep, hist.val_loss,   label='Val', linestyle='--')
    axes[0,i].set_title(f'{name}\nLoss', fontsize=9, fontweight='bold')
    axes[0,i].legend(fontsize=7)
    axes[1,i].plot(ep, hist.train_acc, label='Train')
    axes[1,i].plot(ep, hist.val_acc,   label='Val', linestyle='--')
    axes[1,i].axhline(1-mwpm_ler, color='red', linestyle=':', linewidth=1.5, label=f'MWPM acc={1-mwpm_ler:.3f}')
    axes[1,i].set_title('Accuracy', fontsize=9)
    axes[1,i].legend(fontsize=7)
plt.suptitle('Training curves — v2 models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 5.2 LER сравнение

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#2ca02c' if r['decoder'] != 'MWPM' else '#1f77b4' for _, r in df.iterrows()]
bars = ax.barh(df['decoder'], df['LER'], color=colors, height=0.6)
ax.bar_label(bars, fmt='%.4f', padding=4, fontsize=9)
ax.axvline(mwpm_ler, color='red', linestyle='--', linewidth=1.5, label=f'MWPM={mwpm_ler:.4f}')
ax.set_xlabel('Logical Error Rate (lower is better)')
ax.set_title(f'Decoder Comparison — d={DISTANCE}, p={NOISE_P}', fontweight='bold')
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
plt.show()


## 6. LER vs физическая вероятность ошибки p

In [ ]:
noise_rates = np.array([0.002, 0.005, 0.008, 0.01, 0.015, 0.02, 0.03, 0.05])
N_SWEEP = 5000

sweep_curves = {'MWPM': [], 'AncillaTransformer': [], 'SurfaceCodeCNN': [], 'Ensemble': []}

print('Sweeping noise rates...')
for p in noise_rates:
    sw_cfg = QECConfig(distance=DISTANCE, noise=NoiseConfig(p=p, rounds=ROUNDS))
    sw_gen = SyndromeGenerator(sw_cfg)
    sw_data = sw_gen.generate(n_samples=N_SWEEP)

    sw_mwpm = MWPMDecoder(sw_cfg).build(sw_gen.get_circuit())
    sweep_curves['MWPM'].append(np.mean(sw_mwpm.decode_batch(sw_data.syndromes) != sw_data.logical_errors))
    sweep_curves['AncillaTransformer'].append(np.mean(at_w.decode_batch(sw_data.syndromes) != sw_data.logical_errors))
    sweep_curves['SurfaceCodeCNN'].append(np.mean(cnn_w.decode_batch(sw_data.syndromes) != sw_data.logical_errors))
    ens_p = np.mean(ensemble.decode_batch(sw_data.syndromes) != sw_data.logical_errors)
    sweep_curves['Ensemble'].append(ens_p)
    print(f'  p={p:.3f}: MWPM={sweep_curves["MWPM"][-1]:.4f}, '
          f'Ancilla={sweep_curves["AncillaTransformer"][-1]:.4f}, '
          f'CNN={sweep_curves["SurfaceCodeCNN"][-1]:.4f}, '
          f'Ensemble={ens_p:.4f}')

all_curves = {k: (noise_rates, np.array(v)) for k, v in sweep_curves.items()}
fig = plot_ler_vs_noise(all_curves, title=f'LER vs p — d={DISTANCE}')
plt.show()


## 7. Ablation study — что именно помогло?

Сравниваем вклад каждого улучшения изолированно.

In [ ]:
print('=== Ablation Study ===')
print(f'v1 Transformer + BCE (из прошлого запуска): LER ≈ 0.273')
print(f'v1 Transformer + FocalLoss:                  LER = {trv1_result["logical_error_rate"]:.4f}')
print(f'AncillaTransformer + FocalLoss:              LER = {at_result["logical_error_rate"]:.4f}')
print(f'ResidualMLP + FocalLoss:                     LER = {rmlp_result["logical_error_rate"]:.4f}')
print(f'SurfaceCodeCNN + FocalLoss:                  LER = {cnn_result["logical_error_rate"]:.4f}')
print(f'Ensemble:                                     LER = {ens_ler:.4f}')
print(f'MWPM baseline:                               LER = {mwpm_ler:.4f}')
print()

improvements = [
    ('FocalLoss vs BCE (Transformer-v1)',     0.273 - trv1_result['logical_error_rate']),
    ('2D pos encoding (Ancilla vs v1)',        trv1_result['logical_error_rate'] - at_result['logical_error_rate']),
    ('Residual+features (ResidualMLP vs v1)', 0.328 - rmlp_result['logical_error_rate']),
    ('Physical layout (CNN v2 vs v1)',         0.338 - cnn_result['logical_error_rate']),
    ('Ensemble vs best single model',          min(trv1_result['logical_error_rate'],
                                                    at_result['logical_error_rate']) - ens_ler),
]
print('Contribution of each improvement (LER reduction):')
for name, delta in improvements:
    bar = '█' * max(0, int(delta * 500))
    print(f'  {name:45s}: Δ={delta:+.4f}  {bar}')


## 8. Детальный анализ лучшей модели

In [ ]:
from sklearn.metrics import classification_report
best_preds = at_w.decode_batch(test_raw.syndromes)
labels = test_raw.logical_errors

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_confusion(labels, mwpm_preds, title='MWPM', ax=axes[0])
plot_confusion(labels, best_preds, title='AncillaTransformer', ax=axes[1])
plt.tight_layout()
plt.show()

print('=== AncillaTransformer Classification Report ===')
print(classification_report(labels, best_preds, target_names=['No error', 'Logical error']))


## 9. Выводы

### Диагноз проблем v1 и примененные исправления

| Проблема в v1 | Решение в v2 | Эффект |
|---------------|--------------|--------|
| BCE на несбалансированных классах (64/36) | FocalLoss (α=0.75, γ=2) | Лучший recall для logical error |
| LR=3e-4 → коллапс к мажоритарному классу | LR=1e-4 + 5 эпох warmup | Стабильное обучение с первой эпохи |
| 1D позиционные embeddings в Transformer | 2D row/col + round embeddings (AncillaTransformer) | Модель знает геометрию кода |
| sqrt-паддинг в CNN (5×5 вместо 4×5) | SurfaceCodeCNN: (d-1)×d сетка | Правильный inductive bias |
| 50k шотов — мало для больших моделей | 200k шотов | Лучшая генерализация |
| MLP без структуры | ResidualMLP + syndrome statistics | Явный сигнал плотности ошибок |

### Почему Transformer не превосходит MWPM легко?

MWPM — это **оптимальный** декодер для независимого шума (теорема Деннис 2002).
ML-декодеры могут превзойти его только при:
1. Коррелированном шуме (circuit-level noise)
2. Нестандартных моделях шума, не учтённых MWPM
3. Очень малых расстояниях кода, где MWPM неточен

При `p=0.01` и `d=5` с деполяризующим шумом MWPM близок к оптимуму.
Для соревновательного сравнения используйте `noise_model='circuit_level'`
и расстояния `d=3..9`.
